# 🏛️ MedlinePlus Web Service (NIH / NLM) Explorer
Exploring the official **National Library of Medicine (NLM)** Web Service to power real-time, authoritative medical encyclopedia summaries in our app.

---

### What is the MedlinePlus Web Service?
MedlinePlus is the flagship health information service of the **U.S. National Library of Medicine (NIH)**. They provide a public search Web API that queries their peer-reviewed **Health Topics** database.

### Why is this great for our app?
- **Zero Local Database Required**: No downloading or hosting a 500MB dictionary database.
- **Authoritative & Peer-Reviewed**: High-quality, trusted medical summaries from the NIH.
- **Free & No Registration**: No API keys, no OAuth tokens, and no licensing fees required.
- **Official Summary**: Gives students the clean, high-yield overview of diseases, symptoms, and treatments.

### Key API Specs:
- **Endpoint**: `https://wsearch.nlm.nih.gov/ws/query`
- **Rate Limit**: ~85 requests/minute per IP address.
- **Formats**: Returns structured XML (which we parse into clean JSON).
- **Caching Recommendation**: Cache results for 12–24 hours to stay well within rate limits.

In [2]:
import urllib.request
import urllib.parse
import xml.etree.ElementTree as ET
import html
import re
import json
import time
from typing import List, Dict, Optional
import pandas as pd
from IPython.display import HTML, display
from typing import Dict, Any

MEDLINEPLUS_API_URL = "https://wsearch.nlm.nih.gov/ws/query"

print("✅ Standard libraries imported successfully.")
print(f"API Target: {MEDLINEPLUS_API_URL}")

✅ Standard libraries imported successfully.
API Target: https://wsearch.nlm.nih.gov/ws/query


## 1. Core API Request & XML Parser
The MedlinePlus service returns an XML document with `<nlmSearchResult>` containing `<list>` and `<document>` nodes.

Inside each `<document>`, it provides `<content name="...">` nodes:
- `title`: The official topic name
- `FullSummary` or `snippet`: Plain-language medical summary
- `altTitle`: Common aliases, synonyms, and abbreviations
- `url`: Direct canonical URL on MedlinePlus.gov

Let's write a robust, production-ready Python client to fetch and convert this into clean Python dictionaries.

In [3]:
def clean_xml_text(raw_text: Optional[str]) -> str:
    """Strip embedded HTML tags, decode entities, and normalize whitespace."""
    if not raw_text:
        return ""
    text = re.sub(r"<[^>]+>", "", raw_text)
    text = html.unescape(text)
    return re.sub(r"\s+", " ", text).strip()

def parse_summary_sections(full_summary_html: str) -> List[Dict[str, str]]:
    """Split full-summary HTML by <h3> headers into distinct collapsible sections."""
    if not full_summary_html:
        return []
    sections = []
    # Split on <h3>Heading</h3>
    parts = re.split(r'<h3>(.*?)</h3>', full_summary_html, flags=re.I)
    for i in range(1, len(parts), 2):
        heading = clean_xml_text(parts[i])
        body = parts[i+1].strip()
        sections.append({
            "heading": heading,
            "body": body,
            "clean_text": clean_xml_text(body)
        })
    return sections

def search_medlineplus(
    query: str, 
    retmax: int = 1, 
    rettype: str = "topic",
    timeout_sec: int = 8
) -> List[Dict[str, any]]:
    """
    Query the NLM MedlinePlus Web Service and return structured health topics with collapsible sections.
    """
    params = {
        "db": "healthTopics",
        "term": query.strip(),
        "retmax": str(retmax),
        "rettype": rettype
    }
    
    url = f"{MEDLINEPLUS_API_URL}?{urllib.parse.urlencode(params)}"
    
    req = urllib.request.Request(
        url, 
        headers={"User-Agent": "USMLE-Study-Helper/1.0 (Educational App)"}
    )
    
    try:
        with urllib.request.urlopen(req, timeout=timeout_sec) as response:
            raw_xml = response.read()
    except Exception as e:
        print(f"⚠️ Error fetching from MedlinePlus: {e}")
        return []

    root = ET.fromstring(raw_xml)
    results = []
    
    for doc in root.findall(".//document"):
        item = {
            "title": "",
            "summary": "",
            "url": doc.get("url", ""),
            "alt_titles": [],
            "sections": []
        }
        
        # Check for nested <health-topic> node (in rettype=topic)
        ht = doc.find(".//health-topic")
        if ht is not None:
            item["title"] = ht.get("title", "")
            item["url"] = ht.get("url", item["url"])
            also_called = ht.find("also-called")
            if also_called is not None and also_called.text:
                item["alt_titles"].append(clean_xml_text(also_called.text))
            
            fs = ht.find("full-summary")
            if fs is not None and fs.text:
                raw_summary = fs.text
                item["sections"] = parse_summary_sections(raw_summary)
                item["summary"] = clean_xml_text(raw_summary)
        
        # Fallback to <content> tags
        for content in doc.findall("content"):
            name = content.get("name")
            text = content.text or ""
            
            if name == "title" and not item["title"]:
                item["title"] = clean_xml_text(text)
            elif name in ("FullSummary", "snippet") and not item["summary"]:
                item["summary"] = clean_xml_text(text)
                if not item["sections"] and "<h3>" in text:
                    item["sections"] = parse_summary_sections(text)
            elif name == "altTitle":
                cleaned_alt = clean_xml_text(text)
                if cleaned_alt and cleaned_alt not in item["alt_titles"]:
                    item["alt_titles"].append(cleaned_alt)
                    
        if item["title"]:
            results.append(item)
            
    return results

print("✅ Enhanced MedlinePlus topic & section parser defined.")

✅ Enhanced MedlinePlus topic & section parser defined.


## 2. Real-World Testing: Key USMLE Concepts
Let's test the endpoint with high-yield USMLE medical conditions:
1. `Kawasaki disease`
2. `Aortic stenosis`
3. `Chlamydia infections`
4. `Pheochromocytoma`
5. `Cervicitis`

In [5]:
test_topics = [
    "Kawasaki disease",
    "Aortic stenosis",
    "Chlamydia infections",
    "Pheochromocytoma"
]

for topic in test_topics:
    print("=" * 70)
    print(f"🔍 Searching MedlinePlus for: '{topic}'")
    print("=" * 70)
    
    start = time.time()
    results = search_medlineplus(topic, retmax=1)
    elapsed_ms = (time.time() - start) * 1000
    
    print(f"⏱️ Response Time: {elapsed_ms:.1f}ms | Matches: {len(results)}\n")
    
    for i, res in enumerate(results, 1):
        print(f"[{i}] 📌 {res['title'].upper()}")
        print(f"    🔗 URL: {res['url']}")
        if res['alt_titles']:
            print(f"    🏷️ Also Known As: {', '.join(res['alt_titles'][:4])}")
        print(f"    📖 Summary:")
        # Wrap and indent summary
        summary_snippet = res['summary'][:320] + ("..." if len(res['summary']) > 320 else "")
        print(f"       {summary_snippet}\n")

🔍 Searching MedlinePlus for: 'Kawasaki disease'
⏱️ Response Time: 98.8ms | Matches: 1

[1] 📌 KAWASAKI DISEASE
    🔗 URL: https://medlineplus.gov/kawasakidisease.html
    🏷️ Also Known As: Mucocutaneous lymph node syndrome
    📖 Summary:
       What is Kawasaki disease? Kawasaki disease is a rare illness that usually affects small children. Other names for the disease are Kawasaki syndrome and mucocutaneous lymph node syndrome. It is a type of vasculitis, which is inflammation of the blood vessels. Kawasaki disease is serious, but most children can fully reco...

🔍 Searching MedlinePlus for: 'Aortic stenosis'
⏱️ Response Time: 1066.5ms | Matches: 1

[1] 📌 HEART VALVE DISEASES
    🔗 URL: https://medlineplus.gov/heartvalvediseases.html
    🏷️ Also Known As: Valvular heart disease
    📖 Summary:
       What are heart valve diseases? Heart valve disease happens when one or more of your heart valves don't work well. Your heart has four valves: the tricuspid, pulmonary, mitral, and aortic val

## 3. UI Component Simulation (How it looks in the App Sidebar)
Let's render a preview of how this encyclopedia entry will look inside our React right sidebar or review modal.

In [6]:
from IPython.display import HTML, display
from typing import Dict, Any

def render_encyclopedia_card_html(result: Dict[str, Any]) -> str:
    """
    Generate an interactive collapsible accordion card for Jupyter and Web sidebar.
    - First question/section is open by default (<details open>).
    - Subsequent questions are collapsed vertically (<details>).
    - Lists (<ul><li>) and inline links are cleanly styled.
    """
    if not result:
        return "<p style='color: #64748b;'>No results found.</p>"

    # Badges for alternative names / synonyms
    alt_badges = "".join([
        f"<span style='background: #f1f5f9; color: #475569; padding: 2px 8px; border-radius: 12px; font-size: 11px; margin-right: 4px; display: inline-block; margin-bottom: 4px;'>{t}</span>"
        for t in result.get("alt_titles", [])[:4]
    ])

    sections = result.get("sections", [])
    sections_html = ""

    if sections:
        for idx, sec in enumerate(sections):
            heading = sec["heading"]
            body = sec["body"]
            # First section is expanded by default; subsequent sections are collapsed
            is_open = "open" if idx == 0 else ""

            sections_html += f"""
            <details {is_open} style="margin-bottom: 8px; border: 1px solid #e2e8f0; border-radius: 8px; background: #fafafa; overflow: hidden;">
                <summary style="cursor: pointer; padding: 10px 14px; font-weight: 600; font-size: 13px; color: #1e293b; user-select: none; background: #f8fafc; border-bottom: 1px solid #edf2f7; list-style-position: outside;">
                    {heading}
                </summary>
                <div style="padding: 12px 14px; font-size: 13px; line-height: 1.6; color: #334155; background: #ffffff;">
                    <style>
                        .medline-content ul {{ margin: 8px 0; padding-left: 20px; }}
                        .medline-content li {{ margin-bottom: 4px; }}
                        .medline-content p {{ margin: 6px 0; }}
                        .medline-content a {{ color: #4f46e5; text-decoration: underline; text-underline-offset: 2px; }}
                    </style>
                    <div class="medline-content">
                        {body}
                    </div>
                </div>
            </details>
            """
    else:
        # Fallback if no sub-sections found
        sections_html = f"""
        <div style="font-size: 13px; line-height: 1.6; color: #334155; margin-bottom: 12px;">
            {result.get('summary', '')}
        </div>
        """

    card_html = f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;
                max-width: 520px; border: 1px solid #cbd5e1; border-radius: 14px; padding: 18px;
                background: #ffffff; box-shadow: 0 4px 14px rgba(0,0,0,0.06); margin-top: 10px; margin-bottom: 16px;">
        
        <!-- Header badge & title -->
        <div style="margin-bottom: 10px;">
            <span style="background: #ecfdf5; color: #047857; font-size: 11px; font-weight: 700; padding: 3px 8px; border-radius: 6px; text-transform: uppercase; letter-spacing: 0.5px;">
                🏛️ MedlinePlus • NIH
            </span>
            <h3 style="margin: 8px 0 4px 0; font-size: 18px; font-weight: 700; color: #0f172a;">
                {result.get('title', 'Unknown Topic')}
            </h3>
            {f'<div style="margin-top: 6px;">{alt_badges}</div>' if alt_badges else ''}
        </div>

        <!-- Collapsible sections accordion -->
        <div style="margin-top: 12px; margin-bottom: 14px;">
            {sections_html}
        </div>

        <!-- Footer / Attribution -->
        <div style="border-top: 1px solid #f1f5f9; padding-top: 10px; display: flex; justify-content: space-between; align-items: center;">
            <span style="font-size: 11px; color: #94a3b8;">U.S. National Library of Medicine</span>
            <a href="{result.get('url', '#')}" target="_blank" style="font-size: 12px; font-weight: 600; color: #4f46e5; text-decoration: none;">
                Official Page &rarr;
            </a>
        </div>
    </div>
    """
    return card_html


# --- Run query (retmax=1) and display preview ---
results = search_medlineplus("Kawasaki disease", retmax=1, rettype="topic")

if results:
    display(HTML(render_encyclopedia_card_html(results[0])))
else:
    print("No topic found for query.")

## 4. Proposed FastAPI Endpoint & In-Memory Caching
To connect this to our React frontend:
1. We create `GET /api/v1/encyclopedia?q={term}` in `src/backend/app/main.py`.
2. We add an in-memory cache (e.g. `functools.lru_cache` or a simple TTL dict) so repeated searches for `"Kawasaki disease"` or `"Chlamydia"` are returned in **0.1ms** and never exceed NLM's 85 requests/minute threshold.
3. The React sidebar simply calls this endpoint and renders the clean card!

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=256)
def cached_medlineplus_search(query: str) -> List[Dict[str, any]]:
    """Cached search simulation for our FastAPI backend."""
    return search_medlineplus(query, retmax=3)

# First call: hits the network API
t0 = time.time()
res1 = cached_medlineplus_search("pheochromocytoma")
ms1 = (time.time() - t0) * 1000
print(f"Call 1 (Network fetch): {ms1:.1f}ms - Found {len(res1)} records")

# Second call: served instantly from memory cache!
t1 = time.time()
res2 = cached_medlineplus_search("pheochromocytoma")
ms2 = (time.time() - t1) * 1000
print(f"Call 2 (Cache hit):     {ms2:.3f}ms (Zero network latency!)")